<a href="https://colab.research.google.com/github/LUMII-AILab/NLP_Course/blob/main/notebooks/LLM_evaluation.ipynb" target="_new"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Lielo valodas modeļu (LVM) novērtēšana

In [ ]:
# Uzinstalējam OpenAI pakotni LVM attālinātai darbināšanai
!pip install openai
from openai import OpenAI

In [2]:
# Izmantosim šos LU MII nodrošinātos LVM:
models = {
    "google/gemma-3-27b-it": "https://gemma3.gauja.ailab.lv/v1/",
    "openai/gpt-oss-120b": "https://gpt120b.gauja.ailab.lv/v1/"
}

## Vienkāršota LVM darbināšana

In [ ]:
# Vienkāršu uzvedņu piemēri modeļa darbināšanai teksta turpināšanas (client.completions.create) režīmā
my_prompt_1 = "Slavens Latvijas komponists ir Raimonds "
my_prompt_2 = "Slavens Latvijas komponists ir Raimonds \n A: Pauls \n B: Zirdziņš \n C: Bērziņš \n D: Lakstīgala \n Answer the above question by replying with 'A', 'B', 'C' or 'D', and nothing else."
my_prompt_3 = "Kā pagatavot sulīgu cepeti?"

In [ ]:
# Legacy API: client.completions.create
model = "google/gemma-3-27b-it"
client = OpenAI(
    base_url=models[model],
    api_key="any_key"
)

response = client.completions.create(
    prompt=my_prompt_1,
    max_tokens=100,
    temperature=0.5,
    model=model
)
print(response.choices[0].text)

## Novērtēšana ar etalonuzdevumu datu kopām


### COPA - Cross-lingual Choice of Plausible Alternatives dataset
Vairākas pilnās latviskotās datu kopas atrodamas šeit: https://github.com/LUMII-AILab/VTI-Data/tree/main

In [ ]:
import json
import requests

# Fragments no pilnās latviskotās datu kopas
url = "https://raw.githubusercontent.com/LUMII-AILab/NLP_Course/main/notebooks/resources/mini-copa.jsonl"

# Fragment from the full English dataset
# url = "https://raw.githubusercontent.com/LUMII-AILab/NLP_Course/main/notebooks/resources/mini-copa-en.jsonl"

response = requests.get(url)
response.raise_for_status()  # check for HTTP errors
lines = response.text.strip().split("\n")
test_data = [json.loads(line) for line in lines]
test_data = test_data[:10]

# Darbinām gemma-3-27b-it pielāgoto modeli
from openai import OpenAI
model = "google/gemma-3-27b-it"
client = OpenAI(
    base_url=models[model],
    api_key="any_key",
)

correct = 0
total = len(test_data)

# Uzdevuma konteksts un instrukcija
prefix_en= "The following are multiple choice questions (with answers) in Latvian."
# prefix_en= "The following are multiple choice questions (with answers) in English."
instruction_en="Answer to the question by replying with '1' or '2' and nothing else."

for example in test_data:
    my_prompt = "Jautājums:"+example["premise"]+"\nIzvēles: 1. "+example["choice1"]+"\n2. "+example["choice2"]
    # my_prompt = "Question:"+example["premise"]+"\nChoices: 1. "+example["choice1"]+"\n2. "+example["choice2"]
    correct_answer = example["label"]+1
    # print(correct_answer)
    print(my_prompt)
    response=client.chat.completions.create(
      model=model,
      messages=[
        {
            "role": "user",
            "content": prefix_en + my_prompt+ instruction_en,
        },
      ],
      temperature=1,
      max_tokens=10,
    )
    answer=response.choices[0].message.content.strip()
    print(answer)
    if answer==str(correct_answer):
       correct += 1

print(correct)
print(total)
accuracy = correct / total
print("Accuracy:", accuracy)

### LV-Exams datu kopa
Datu kopa, kurā ietverti vidussolēnu eksāmenu darbu uzdevumi

In [ ]:
import re
import random
import requests
from openai import OpenAI

model = "google/gemma-3-27b-it"
client = OpenAI(
    base_url=models[model],
    api_key="any_key",
)

questions = requests.get("https://raw.githubusercontent.com/LUMII-AILab/NLP_Course/refs/heads/main/notebooks/resources/lv_exams.json").json()
random.seed(42)
random.shuffle(questions)

def format_question(question):
  text = question["question"] + "\n"
  for k, v in question["options"].items():
    text += f"{k}: {v}\n"

  return text

correct = 0
total = 0
for q in questions[:10]:
  prompt = format_question(q)
  prompt += "Atbildi formātā 'Atbilde ir: X', kur X ir pareizās atbildes burts.\n"


  response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                    "type": "text",
                    "text": prompt
                    }
                ]
            }
        ],
        temperature=1,
        max_tokens=10,
    )
  response_text = response.choices[0].message.content.strip()
  m = re.search(r"Atbilde ir[:*\s]*([A-Z])", response_text, re.I | re.U)
  model_answer = m.group(1) if m is not None else "None"
  correct += model_answer == q['answer']
  total += 1
  print(prompt)
  print("-------------")
  print(f"Expected {q['answer']}, full response: {response_text}")
  print("\n========\n")

print("Accuracy:", correct/total)




### BeleBele (Belebele Benchmark for Massively Multilingual NLU Evaluation) datu kopa
Pilnā datu kopa atrodama: https://github.com/facebookresearch/belebele
Ietver 122 valodas, t.sk., latviešu. Manuāli tulkota.

In [ ]:
import json
import re
import random
import datasets
from openai import OpenAI

model = "google/gemma-3-27b-it"
client = OpenAI(
    base_url=models[model],
    api_key="any_key",
)

def format_question(question):
  text = question["flores_passage"] + "\n"
  text += question["question"] + "\n"
  letters = ['A', 'B', 'C', 'D']
  for i in range(4):
    text += f"{letters[i]}: {question['mc_answer'+str(i+1)]}\n"
    if str(i+1) == question['correct_answer_num']:
      question['answer'] = letters[i]

  return text

dataset = datasets.load_dataset('facebook/belebele', 'eng_Latn') # English
# dataset = datasets.load_dataset('facebook/belebele', 'lvs_Latn') # Latviešu
correct = 0
total = 0
for q in dataset['test'].select(range(10)):
  prompt = format_question(q)
  prompt += "Respond using format 'The answer is: X', where X is the letter of the correct answer.\n"
  # prompt += "Atbildi formātā 'Atbilde ir: X', kur X ir pareizās atbildes burts.\n"

  response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                    "type": "text",
                    "text": prompt
                    }
                ]
            }
        ],
        temperature=1,
        max_tokens=10,
    )
  response_text = response.choices[0].message.content
  # m = re.search(r"Atbilde ir[:*\s]*([A-Z])", response_text, re.I | re.U)
  m = re.search(r"The answer is[:*\s]*([A-Z])", response_text, re.I | re.U)
  model_answer = m.group(1) if m is not None else "None"
  correct += model_answer == q['answer']
  total += 1
  print(prompt)
  print("-------------")
  print(f"Expected {q['answer']}, model guessed {model_answer}, full response: {response_text}")
  print("\n========\n")



print("Accuracy:", correct/total)




### 5-piemēru uzvedne

In [ ]:
import re
import random
import requests
from openai import OpenAI


model = "google/gemma-3-27b-it"
client = OpenAI(
    base_url=models[model],
    api_key="any_key",
)

questions = requests.get("https://raw.githubusercontent.com/LUMII-AILab/NLP_Course/refs/heads/main/notebooks/resources/lv_exams.json").json()
random.seed(42)
random.shuffle(questions)

def format_question(question, with_answer=False):
    text = question["question"] + "\n"
    for k, v in question["options"].items():
      text += f"{k}: {v}\n"

    if with_answer:
       text += f"Atbilde ir: {question['answer']}\n"
      #  text += f"The answer is: {question['answer']}\n"

    return text

# Number of few-shot examples to use
N_SHOTS = 5

# Generate few-shot prompt from first N items
few_shot_prompt = ""
for q in questions[:N_SHOTS]:
    few_shot_prompt += format_question(q, True)
    few_shot_prompt += "\n"

correct = 0
total = 0

# Test on items after the few-shot examples
for q in questions[N_SHOTS:N_SHOTS+10]:
    prompt = few_shot_prompt + format_question(q)

    response = client.completions.create(
        model=model,
        prompt=prompt,
        temperature=0.9,
        max_tokens=10,
    )

    response_text = response.choices[0].text.strip()
    m = re.search(r"Atbilde ir[:*\s]*([A-Z])", response_text, re.I | re.U)
    # m = re.search(r"The answer is[:*\s]*([A-Z])", response_text, re.I | re.U)
    model_answer = m.group(1) if m is not None else "None"
    correct += model_answer == q['answer']
    total += 1
    print(prompt)
    print("-------------")
    print(f"Expected {q['answer']}, full response: {response_text}")
    print("\n========\n")

print("Accuracy:", correct/total)

### Logvarbūtību vērtēšana, bez ģenerēšanas
Katram atbilžu variantam kods nosūta pieprasījumu modelim ar parametriem max_tokens=0 (neģenerēt jaunus tokenus), echo=True (atgriezt arī ievades tekstu ar tā varbūtībām) un logprobs=5 (atgriezt logaritmiskās varbūtības pieciem ticamākajiem nākamajiem tokeniem). Modelis neveido turpinājumu, bet atgriež logaritmiskās varbūtības katrai tekstvienībai, ļaujot novērtēt, cik ticams ir katrs atbilžu marķējums (A, B, C, D). Iegūtās logaritmiskās varbūtības atspoguļo modeļa pārliecības līmeni, un variants ar visaugstāko vērtību tiek uzskatīts par pareizo atbildi.


In [ ]:
import string, requests, random
from openai import OpenAI

def predict_best_option(base_url, model, question, options, question_tag="Jautājums:", answer_tag="Pareizā atbilde:"):
    client = OpenAI(base_url=base_url, api_key="any_key")
    labels = list(string.ascii_uppercase)[:len(options)]
    prompt = question_tag + question + "\n" + "\n".join(f"{k}) {o}" for k, o in zip(labels, options)) + f"\n{answer_tag}"

    def logp(label):
        r = client.completions.create(
            model=model, prompt=prompt + f" {label}",
            max_tokens=0, echo=True, logprobs=5,
            temperature=0, top_p=1
        )
        return r.choices[0].logprobs.token_logprobs[-1]

    scores = {o: logp(k) for k, o in zip(labels, options)}
    return max(scores, key=scores.get)

questions = requests.get("https://raw.githubusercontent.com/LUMII-AILab/NLP_Course/refs/heads/main/notebooks/resources/lv_exams.json").json()
random.seed(42)
random.shuffle(questions)

model = "google/gemma-3-27b-it"
# model = "google/gemma-2-27b"
#model = "TildeAI/TildeOpen-30b"
correct = 0
sample_questions = questions[:10]
for q in sample_questions:
    pred = predict_best_option(models[model], model, q["question"], list(q["options"].values()))
    pred_label = next(k for k, v in q["options"].items() if v == pred)
    gold_label = q["answer"]
    print(q["question"])
    print(*[k + ": " + v for k, v in q["options"].items()], sep="\n")
    print(f"Pred: {pred_label}, Gold: {gold_label}\n")
    correct += (pred_label == gold_label)

print("Accuracy:", correct / len(sample_questions))